# Week 4: Build a Tiny LLM From Scratch

Last week ended at self-attention: one query, one key, one value projection, computed by hand and then in code as the `SelfAttention` class. That class was deliberately built to mirror the shape multi-head attention will need, project, score, softmax, weighted sum, while leaving out exactly two things: splitting the projection into multiple heads, and mixing those heads back together afterward.

This week starts by adding those two things. From there we build everything Week 3 didn't get to: causal masking, a full transformer block, and a trainable model, then actually train it on real text.


## 1. Bringing Week 3's Code

Three pieces come forward unchanged, exactly what you built last week.


### 1.1 Positional encoding

The sine/cosine function for Positional Encoding from Week 3.


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)
np.random.seed(0)

def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    position = np.arange(seq_len)[:, np.newaxis]
    i = np.arange(d_model // 2)
    div_term = 10000 ** (2 * i / d_model)
    pe[:, 0::2] = np.sin(position / div_term)
    pe[:, 1::2] = np.cos(position / div_term)
    return pe

pe_check = positional_encoding(seq_len=3, d_model=4)
print("PE(1, :) =", pe_check[1].round(4), " matches Week 3's hand calc: [0.8415, 0.5403, 0.0100, 0.99995]")


PE(1, :) = [0.8415 0.5403 0.01   1.    ]  matches Week 3's hand calc: [0.8415, 0.5403, 0.0100, 0.99995]


### 1.2 Greedy and Sampled Decoding

From Week 3, we are bringing in Greedy and Sampled Decoding.


In [2]:
def softmax_np(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

def greedy_decode(logits, vocab):
    return vocab[np.argmax(logits)]

def sample_decode(logits, vocab, temperature=1.0, rng=None):
    rng = rng or np.random.default_rng()
    probs = softmax_np(logits / temperature)
    return rng.choice(vocab, p=probs), probs

vocab_toy = ["cat", "dog", "ran", "slept", "quickly"]
logits_toy = np.array([2.5, 2.3, 1.0, 0.5, -0.2])
print("greedy pick:", greedy_decode(logits_toy, vocab_toy))


greedy pick: cat


### 1.3 The `SelfAttention` class

The exact class from Week 3, including the same toy "dog / squirrel / it" example and the same hand-picked $W_Q, W_K, W_V$ matrices that broke the tie between "dog" and "squirrel" (0.807 vs. 0.097). We'll build on this directly.


In [3]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.d_k = d_k
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)

    def forward(self, x):
        batch, seq_len, d_model = x.shape
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)
        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        output = weights @ V
        return output, weights

# same toy tokens and hand-picked matrices as Week 3
x_dog      = torch.tensor([1., 0., 0., 1.])
x_squirrel = torch.tensor([0., 1., 0., 1.])
x_it       = torch.tensor([0., 0., 1., 1.])
x = torch.stack([x_dog, x_squirrel, x_it]).unsqueeze(0)  # (batch=1, seq_len=3, d_model=4)

attn = SelfAttention(d_model=4, d_k=2)
with torch.no_grad():
    attn.W_Q.weight.copy_(torch.tensor([[0.,0.],[0.,0.],[3.,0.],[0.,0.]]).T)
    attn.W_K.weight.copy_(torch.tensor([[1.,0.],[0.,1.],[0.,0.],[0.,0.]]).T)
    attn.W_V.weight.copy_(torch.tensor([[1.,0.],[0.,1.],[0.,0.],[0.,0.]]).T)

output, weights = attn(x)
print("Attention weights for 'it' -> [dog, squirrel, it]:", [round(w, 4) for w in weights[0, 2].tolist()])


Attention weights for 'it' -> [dog, squirrel, it]: [0.8066, 0.0967, 0.0967]


## 2. Multi-Head Attention

### Why one head isn't enough

`SelfAttention` gives "it" exactly one shot at deciding what matters: one $W_Q$, one $W_K$, one $W_V$, one resulting relevance pattern. But real sentences usually have more than one relationship worth tracking at once. In our toy example, $W_Q$ and $W_K$ were hand-picked to solve one specific question: does "it" refer to "dog" or "squirrel"? A single head can be shaped to answer that question well, but the same set of weights now also has to handle every other kind of relationship a token might need, subject-verb agreement, tense, which adjective modifies which noun, and so on. One projection, one shared 2-number space, can only prioritize so much at once before different relationships start to interfere with each other.

The fix is not a smarter single head, it's several heads running in parallel, each with its own $W_Q$, $W_K$, $W_V$, each free to specialize in a different kind of relationship, combined only at the very end.

### From `SelfAttention` to `MultiHeadAttention`

Instead of writing several separate `SelfAttention` modules and concatenating their outputs, which would work but wastes computation, we do the mathematically equivalent thing in one step: project to the full `d_model` size once, then reshape that projection into `n_heads` separate chunks. Concretely, `MultiHeadAttention` adds exactly two things on top of `SelfAttention`:

1. **Head splitting.** `W_Q`, `W_K`, `W_V` now project from `d_model` to `d_model` (not down to a smaller `d_k`), and `.view(...).transpose(...)` reshapes that single projection into `n_heads` chunks of size `d_k = d_model // n_heads` each, giving every head its own slice of the space to work in.
2. **A final mixing layer, `W_O`.** After each head computes its own attention output, the heads get concatenated back into a `d_model`-sized vector, then passed through one more linear layer that blends the heads' separate findings into a single combined representation. Without `W_O`, the heads' outputs would just sit side by side, unmixed.


In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must divide evenly into n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal_mask=None):
        batch, seq_len, d_model = x.shape

        Q = self.W_Q(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        if causal_mask is not None:
            scores = scores.masked_fill(causal_mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        head_outputs = weights @ V

        concat = head_outputs.transpose(1, 2).contiguous().view(batch, seq_len, d_model)
        return self.W_O(concat), weights

# same toy sequence, now with 2 heads instead of 1
mha = MultiHeadAttention(d_model=4, n_heads=2)
out, attn_weights = mha(x)
print("output shape:", out.shape, " (matches SelfAttention's expected shape, just with n_heads=1 collapsing back to it)")
print("attention weight shape:", attn_weights.shape, " -> (batch, n_heads, seq_len, seq_len), one weight matrix per head")


output shape: torch.Size([1, 3, 4])  (matches SelfAttention's expected shape, just with n_heads=1 collapsing back to it)
attention weight shape: torch.Size([1, 2, 3, 3])  -> (batch, n_heads, seq_len, seq_len), one weight matrix per head


Notice `causal_mask` already appears as a parameter in `forward`, unused so far (it defaults to `None`). That's deliberate, we build the actual mask next.


## 3. Causal Masking

Every attention example so far let "it" look at every other token freely, which was fine for a hand-picked example built to illustrate one idea. But a real model generates left to right, one new token at a time, and during training it processes an entire sequence in parallel for speed. Without a mask, position 3 could peek at position 4's key and value while learning to predict position 4, which means the loss signal wouldn't reflect genuine prediction at all, it would reflect an open-book test.

A causal mask is simple: a lower-triangular matrix of 1s and 0s. Row $t$ has 1s in columns $0$ through $t$ (positions the token can see) and 0s afterward (positions it can't).


In [5]:
seq_len = 3
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
print("causal mask:\n", causal_mask)


causal mask:
 tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])


Let's prove it changes something. Run the same toy input through `MultiHeadAttention` with and without the mask, and look at position 0's attention weights, the very first token, "dog".


In [6]:
_, weights_unmasked = mha(x)
_, weights_masked = mha(x, causal_mask=causal_mask)

print("position 0's attention weights, no mask:  ", weights_unmasked[0, 0, 0].detach().numpy().round(4))
print("position 0's attention weights, with mask:", weights_masked[0, 0, 0].detach().numpy().round(4))


position 0's attention weights, no mask:   [0.3166 0.356  0.3274]
position 0's attention weights, with mask: [1. 0. 0.]


Without the mask, position 0 spreads attention across all three positions, including two that haven't "happened" yet from its point of view. With the mask, its weight collapses entirely onto position 0, the only token it's allowed to see. This is the exact mechanism applied to every row of the score matrix at once: row $t$ gets masked to only ever attend to columns $0 \ldots t$.


## 4. Assembling the Full Transformer Block

Attention alone isn't a transformer block. The original architecture wraps it with two more ingredients:

- **Residual connections**: `x = x + attn(x)` instead of `x = attn(x)`. This gives gradients a direct path backward through the network (helpful once we stack many blocks), and means each block only has to learn a *correction* to its input rather than reconstruct the whole representation from nothing.
- **Layer normalization**: rescales each token's vector to have stable mean and variance before it enters attention or the feedforward step. This keeps training numerically stable as we stack blocks deeper.

Alongside attention, each block also has a small position-wise feedforward network, a two-layer MLP applied identically to every position, giving the model extra capacity to transform each token's representation after attention has mixed in context from its neighbors.


In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, causal_mask):
        attn_out, _ = self.attn(self.ln1(x), causal_mask=causal_mask)
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x

# shape check: a block's output must match its input, so blocks can stack
block_check = TransformerBlock(d_model=8, n_heads=2)
mask_check = torch.tril(torch.ones(3, 3))
out = block_check(torch.randn(1, 3, 8), mask_check)
print("block output shape:", out.shape, "(matches input shape, as required for stacking)")


block output shape: torch.Size([1, 3, 8]) (matches input shape, as required for stacking)


## 5. `TinyTransformerLM`: Putting It All Together

Now every piece exists: token embeddings, Week 3's positional encoding, `MultiHeadAttention` wrapped in a causally-masked `TransformerBlock`, and a final linear "LM head" that projects back to vocabulary-sized logits. `TinyTransformerLM` stacks several blocks and wires all of it together.

The positional encoding is added once, right after the token embedding, using Week 3's function directly. The causal mask is built once, sized to the longest sequence the model will ever see (`block_size`), and sliced down to whatever length the current input actually is.


In [8]:
class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=3, block_size=32):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)

        pe = positional_encoding(block_size, d_model)  # Week 3's function
        self.register_buffer("pos_emb", torch.tensor(pe, dtype=torch.float32))

        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.block_size = block_size

        self.register_buffer("causal_mask", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_emb(idx)
        pos = self.pos_emb[:T].unsqueeze(0)
        x = tok + pos

        mask = self.causal_mask[:T, :T]
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

# shape and gradient smoke test before we touch real data
toy_model = TinyTransformerLM(vocab_size=59, d_model=64, n_heads=4, n_layers=3, block_size=32)
xb_check = torch.randint(0, 59, (16, 32))
yb_check = torch.randint(0, 59, (16, 32))
logits, loss = toy_model(xb_check, yb_check)
print("logits shape:", logits.shape, " loss:", round(loss.item(), 4))
print("total trainable parameters:", sum(p.numel() for p in toy_model.parameters()))


logits shape: torch.Size([16, 32, 59])  loss: 4.2347
total trainable parameters: 156923
